In [1]:
# IMPORT SEMUA LIBRARY YANG DIBUTUHKAM
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse
from IPython.display import display
import os

print("Library berhasil diimpor.")


Library berhasil diimpor.


In [2]:
# SIMPAN URL API
TARGET = "https://satudata.bandungkab.go.id/api/dataset/DISPAREKRAF/jumlah-wisatawan"
UA = "INSTIKI-BIFP243-Praktikum/1.0 (mahasiswa@instiki.ac.id)"
TIMEOUT = 20

headers = {"User-Agent": UA}

print("Target:", TARGET)

Target: https://satudata.bandungkab.go.id/api/dataset/DISPAREKRAF/jumlah-wisatawan


In [3]:
# CEK ROBOTS.TXT
def cek_kepatuhan(url, user_agent=UA):
    parsed = urlparse(url)
    base = f"{parsed.scheme}://{parsed.netloc}"
    robots_url = base + "/robots.txt"

    try:
        response = requests.get(
            robots_url,
            headers={"User-Agent": user_agent},
            timeout=TIMEOUT
        )

        if response.status_code == 200:
            rp = RobotFileParser()
            rp.set_url(robots_url)
            rp.parse(response.text.splitlines())

            return {
                "robots_url": robots_url,
                "status_robots": response.status_code,
                "boleh_diakses": rp.can_fetch(user_agent, url),
                "crawl_delay": rp.crawl_delay(user_agent),
                "isi_robots_txt": response.text[:1000]
            }

        return {
            "robots_url": robots_url,
            "status_robots": response.status_code,
            "boleh_diakses": None,
            "crawl_delay": None,
            "isi_robots_txt": response.text[:1000],
            "catatan": "robots.txt tidak tersedia dengan status 200; periksa ketentuan situs."
        }

    except requests.RequestException as e:
        return {
            "robots_url": robots_url,
            "boleh_diakses": None,
            "error": str(e)
        }


hasil_robots = cek_kepatuhan(TARGET)

for key, value in hasil_robots.items():
    print(f"{key}: {value}\n")


robots_url: https://satudata.bandungkab.go.id/robots.txt

status_robots: 200

boleh_diakses: True

crawl_delay: None

isi_robots_txt: # As a condition of accessing this website, you agree to abide by the following
# content signals:

# (a)  If a Content-Signal = yes, you may collect content for the corresponding
#      use.
# (b)  If a Content-Signal = no, you may not collect content for the
#      corresponding use.
# (c)  If the website operator does not include a Content-Signal for a
#      corresponding use, the website operator neither grants nor restricts
#      permission via Content-Signal with respect to the corresponding use.

# The content signals and their meanings are:

# search:   building a search index and providing search results (e.g., returning
#           hyperlinks and short excerpts from your website's contents). Search does not
#           include providing AI-generated search summaries.
# ai-input: inputting content into one or more AI models (e.g., retrieval
# 

In [4]:
# TEST LAKUKAN GET PADA ENDPOINT API
if hasil_robots.get("boleh_diakses") is False:
    raise PermissionError(
        "robots.txt tidak mengizinkan URL ini untuk User-Agent yang digunakan. "
        "Jangan lanjutkan scraping sebelum mendapat izin atau memastikan ketentuannya."
    )

try:
    r = requests.get(TARGET, headers=headers, timeout=TIMEOUT)
    print("HTTP Status:", r.status_code)
    print("Content-Type:", r.headers.get("Content-Type"))
    r.raise_for_status()
except requests.RequestException as e:
    print("Request gagal:", e)
    raise


HTTP Status: 200
Content-Type: application/json


In [5]:
# CEK TIPE DATA (JSON/HTML)
content_type = r.headers.get("Content-Type", "").lower()

try:
    data = r.json()
    print("Respons berhasil dibaca sebagai JSON.")
    print("Tipe objek:", type(data).__name__)
except ValueError:
    data = None
    print("Respons bukan JSON yang valid.")
    print("Cuplikan respons:")
    print(r.text[:1500])

    # Diagnostik HTML dengan BeautifulSoup
    soup = BeautifulSoup(r.text, "lxml")
    title = soup.title.get_text(strip=True) if soup.title else "Judul tidak ditemukan"
    print("\nJudul halaman:", title)
    print("Jumlah tag <div>:", len(soup.find_all("div")))


Respons berhasil dibaca sebagai JSON.
Tipe objek: dict


In [6]:
# MENAMPILKAN DATA DALAM BENTUK JSON
if isinstance(data, dict):
    print("Key utama:", list(data.keys()))
    print("\nCuplikan JSON:")
    print(json.dumps(data, indent=2, ensure_ascii=False)[:5000])
elif isinstance(data, list):
    print("Jumlah elemen pada list utama:", len(data))
    print("\nContoh elemen pertama:")
    if data:
        print(json.dumps(data[0], indent=2, ensure_ascii=False))
else:
    print("Belum ada objek JSON yang dapat diperiksa.")


Key utama: ['code', 'component', 'category', 'scope', 'frequency', 'activity', 'concept', 'classification', 'definition', 'size', 'unit', 'descr', 'data_column', 'id_metadata', 'series_col_data', 'series_data', 'created_at', 'updated_at', 'sectoral_name', 'chart', 'pd_id', 'pd_name', 'pd_initials', 'status_public', 'programs', 'file', 'dynamics', 'dna', 'data_row', 'table_dynamic']

Cuplikan JSON:
{
  "code": "DISPAREKRAF/001",
  "component": "Jumlah wisatawan",
  "category": "1",
  "scope": "Desa",
  "frequency": "1",
  "activity": "Kompilasi Produk Administrasi",
  "concept": "[K01008] Kunjungan Wisata\r\n[K02331] Wisatawan\r\n[K02332] Wisatawan Mancanegara\r\n[K02334] Wisatawan Nusantara",
  "classification": "[32020026] Wilayah",
  "definition": "[K01008] Tinggal di suatu tempat yang dikunjungi selama perjalanan wisata. Tinggal yang dimaksud tidak perlu bermalam, namun harus berhenti minimal 6 jam untuk memenuhi syarat sebagai kunjungan wisata. Memasuki wilayah geografis tanpa berh

In [7]:
# Mengambil nama kolom dan isi data dari JSON
if data is not None:
    kolom = data["data_column"]
    baris = data["data_row"]

    # Membentuk DataFrame dari data API
    df = pd.DataFrame(baris, columns=kolom)

    # Menghapus spasi berlebih pada nama kolom
    df.columns = df.columns.str.strip()

    # Menampilkan informasi dataset
    print("Data berhasil ditemukan!")
    print("Jumlah record:", len(df))
    print("Jumlah kolom:", len(df.columns))

    # Menampilkan 150 data pertama
    # display(df.head(150))
    display(df)

else:
    df = None
    print("Data JSON tidak ditemukan.")

Data berhasil ditemukan!
Jumlah record: 1328
Jumlah kolom: 13


,No,Kode Wilayah,Wilayah,Tahun,Asal Wisatawan,Kode Kecamatan,Kecamatan,Kode Desa/Kelurahan,Desa/Kelurahan,Bidang Usaha Pariwisata,Nama Usaha Pariwisata,Alamat,Jumlah Kunjungan Wisatawan
0,1,32.04,Kabupaten Bandung,2021,Nusantara,-,-,-,-,-,*,*,3.880.500
1,2,32.04,Kabupaten Bandung,2021,Mancanegara,-,-,-,-,-,*,*,100
2,3,32.04,Kabupaten Bandung,2022,Nusantara,-,-,-,-,-,*,*,6.548.815
3,4,32.04,Kabupaten Bandung,2022,Mancanegara,-,-,-,-,-,*,*,1.748
4,5,32.04,Kabupaten Bandung,2023,Nusantara,-,-,-,-,-,*,*,6.986.976
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,1324,32.04,Kabupaten Bandung,2025,Mancanegara,32.04.06,CIMENYAN,-,-,MAKAN DAN MINUM,*** Ba**,*,222
1324,1325,32.04,Kabupaten Bandung,2025,Mancanegara,32.04.06,CIMENYAN,-,-,MAKAN DAN MINUM,Co****** re*** * Ca** Fa***** Ga****,*,16
1325,1326,32.04,Kabupaten Bandung,2025,Mancanegara,32.04.06,CIMENYAN,-,-,MAKAN DAN MINUM,li**** re***,*,8
1326,1327,32.04,Kabupaten Bandung,2025,Mancanegara,32.04.05,CILEUNYI,-,-,MAKAN DAN MINUM,Ru*** Ma*** Am****,*,10


In [8]:
# Menyimpan DataFrame ke file CSV
folder = '../data/raw/'
nama_file = os.path.join(folder, "data_kunjungan_wisata_kabupaten_bandung-16-nov-26.csv")

df.to_csv(
    nama_file,
    index=False,
    encoding="utf-8-sig"
)

print(f"Data berhasil disimpan ke: {nama_file}")

Data berhasil disimpan ke: ../data/raw/data_kunjungan_wisata_kabupaten_bandung-16-nov-26.csv
